<a href="https://colab.research.google.com/github/plrushe/Honours-Code/blob/main/DISTILBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U "transformers==4.44.2" "datasets==2.21.0" "evaluate==0.4.2" "accelerate==0.33.0"
!pip -q install -U "scikit-learn==1.4.2" "pandas==2.2.2" "numpy==1.26.4"

In [ ]:
import os
import re
import numpy as np
import pandas as pd

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

print("torch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

set_seed(42)

torch: 2.10.0+cu128
device: cuda


In [ ]:
dataset = pd.read_csv("patient_diaries.csv")

dataset = dataset[["text", "label"]].copy()

norm = dataset["label"].astype(str).str.strip().str.lower()
mapping = {"no depression": 0, "depression": 1}
dataset["label"] = norm.map(mapping)

dataset = dataset.dropna(subset=["text", "label"])
dataset["label"] = dataset["label"].astype("int64")
dataset["text"] = dataset["text"].astype(str)

print("Dataset shape:", dataset.shape)
print(dataset["label"].value_counts())
dataset.head()

Dataset shape: (1000, 2)
label
0    514
1    486
Name: count, dtype: int64


,text,label
0,I felt happy today. I shared a laugh with a fr...,0
1,I felt fine today. I wrote in my journal and e...,1
2,I felt confused today. I felt overwhelmed with...,1
3,I felt confused today. I felt a glimmer of hop...,0
4,I felt down today. I felt a glimmer of hope to...,0


In [ ]:
def basic_clean(text: str) -> str:
    text = text.strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

dataset["text"] = dataset["text"].apply(basic_clean)

In [ ]:
train_df, test_df = train_test_split(
    dataset,
    test_size=0.2,
    stratify=dataset["label"],
    random_state=42
)

print("Train:", train_df.shape, "Test:", test_df.shape)

Train: (800, 2) Test: (200, 2)


In [ ]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

# Rename label column to 'labels' (Trainer expects this)
train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Set torch format for Trainer
cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)

training_args = TrainingArguments(
    output_dir="distilbert_out",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,

    logging_steps=50,
    report_to="none",
    seed=42,
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,          # evaluates each epoch on test split
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.407600,0.298283,0.890000,0.921348,0.845361,0.881720
2,0.206600,0.257628,0.885000,0.974359,0.783505,0.868571
3,0.178600,0.211842,0.905000,0.906250,0.896907,0.901554
4,0.169200,0.213894,0.910000,0.891089,0.927835,0.909091


TrainOutput(global_step=200, training_loss=0.24049399852752684, metrics={'train_runtime': 46.7314, 'train_samples_per_second': 68.476, 'train_steps_per_second': 4.28, 'total_flos': 18081799916544.0, 'train_loss': 0.24049399852752684, 'epoch': 4.0})

In [ ]:
pred_output = trainer.predict(test_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids
y_pred = np.argmax(logits, axis=-1)

print("Hold-out test results")
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1:", f1_score(y_true, y_pred, zero_division=0))

print("\nClassification report:\n", classification_report(y_true, y_pred, digits=4))
print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred))

Hold-out test results
Accuracy: 0.91
Precision: 0.8910891089108911
Recall: 0.9278350515463918
F1: 0.9090909090909091

Classification report:
               precision    recall  f1-score   support

           0     0.9293    0.8932    0.9109       103
           1     0.8911    0.9278    0.9091        97

    accuracy                         0.9100       200
   macro avg     0.9102    0.9105    0.9100       200
weighted avg     0.9108    0.9100    0.9100       200


Confusion matrix:
 [[92 11]
 [ 7 90]]


In [ ]:
SAVE_DIR = "models/distilbert_depression_classifier"
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved to:", SAVE_DIR)

Saved to: models/distilbert_depression_classifier
